In [1]:
#Bu notebookda kameranın hareketli olmaası üzerine çalışılmıştır.
import os

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/zlemdemir/modeldosyasi/operator_shadowing_best.pt
/kaggle/input/datasets/zlemdemir/fabrika-video/fabrika_video.mp4


In [3]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 464.6 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.5 MB/s eta 0:00:00


In [6]:
from ultralytics import YOLO

# Modeli yükleme
model = YOLO('/kaggle/input/datasets/zlemdemir/modeldosyasi/operator_shadowing_best.pt')

# Tracking yapma
results = model.track(
    source='/kaggle/input/datasets/zlemdemir/fabrika-video-3/fabrika_video_3.mp4',
    tracker='bytetrack.yaml',
    save=True,
    conf=0.5,
    project='/kaggle/working',
    name='tracking_sonuc'
)

print("\nTracking tamamlandı!")
import os
sonuc_klasoru = '/kaggle/working/tracking_sonuc'
for f in os.listdir(sonuc_klasoru):
    print(f'{sonuc_klasoru}/{f}')


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/253) /kaggle/input/datasets/zlemdemir/fabrika-video-3/fabrika_video_3.mp4: 384x640 2 persons, 67.8ms
video 1/1 (frame 2/253) /kaggle/input/datasets/zlemdemir/fabrika-video-3/fabrika_video_3.mp4: 384x640 2 persons, 65.6ms
video 1/1 (frame 3/253) /kaggle/input/datasets/zlemdemir/fabrika-video-3/fabrika_video_3.mp4: 384x640 2 persons, 66.7ms
video 1/1 (frame 4/253) /kaggle/input/datasets/zlemdemir/fabrika-video-3/fabrika_video_3.mp4: 384x

In [8]:
import pandas as pd

koordinatlar = []

for frame_idx, result in enumerate(results):
    if result.boxes.id is not None:  
        boxes = result.boxes.xywh.cpu().numpy()  # x_merkez, y_merkez, genişlik, yükseklik
        ids = result.boxes.id.cpu().numpy().astype(int)
        classes = result.boxes.cls.cpu().numpy().astype(int)
        
        for box, track_id, cls in zip(boxes, ids, classes):
            x_center, y_center, w, h = box
            koordinatlar.append({
                'frame': frame_idx,
                'id': int(track_id),
                'class': 'person' if cls == 0 else 'high-vis',
                'x': float(x_center),
                'y': float(y_center),
                'width': float(w),
                'height': float(h)
            })

# csv olarak koordinatları kaydettik
df = pd.DataFrame(koordinatlar)
df.to_csv('/kaggle/working/koordinatlar.csv', index=False)

print(f"Toplam {len(df)} koordinat kaydedildi.")
print(f"\nBenzersiz ID sayısı: {df['id'].nunique()}")
print(f"Toplam frame sayısı: {df['frame'].max() + 1}")
print("\nİlk 10 satır:")
print(df.head(10))

Toplam 271 koordinat kaydedildi.

Benzersiz ID sayısı: 4
Toplam frame sayısı: 143

İlk 10 satır:
   frame  id   class           x           y      width     height
0      0   1  person  153.867950  126.860916  38.258118  88.283417
1      0   2  person  200.028946  109.257080  29.171600  76.732178
2      1   1  person  153.988861  126.897156  38.434143  88.706367
3      1   2  person  200.351089  109.229172  29.278809  76.994751
4      2   1  person  154.689209  126.539513  38.123184  87.902115
5      2   2  person  200.752762  108.941299  29.126343  76.557732
6      3   1  person  155.471344  126.856888  38.447037  88.476135
7      3   2  person  201.100815  107.870605  29.903717  78.812996
8      4   1  person  155.967377  127.368546  38.602737  88.662086
9      4   2  person  201.584900  108.509201  30.256943  80.076721


In [9]:
# 4 ve 5 id li personlar ID kaybolmasından dolayı olmus, onları siliyoruz
gecerli_idler = df.groupby('id').size()
gecerli_idler = gecerli_idler[gecerli_idler >= 5].index
df_temiz = df[df['id'].isin(gecerli_idler)]
df_temiz.to_csv('/kaggle/working/koordinatlar_temiz.csv', index=False)
print(f"Temizleme sonrası: {len(df_temiz)} koordinat, {df_temiz['id'].nunique()} ID")

Temizleme sonrası: 268 koordinat, 2 ID
